In [1]:
from sym_graphs.utils.utils import yaml_to_dotdict
from sym_graphs.data_io.dataset import DataLoader
import torch 
from sym_graphs.models.simple_model import DirectQDAGerModel
from sym_graphs.models.sinkhorn_model import SinkhornQDAGerModel
import math 
import numpy as np 

In [2]:

#Names = ["aids","ogbg-molhiv","ogbg-molpcba","yeast","mutagenicity","ogbg-code2","linux"]
Names = ["yeast"]

data_types = ["equal","unequal"]
model_types = ["dir","SH"]

In [ ]:
def get_mean_and_SE(name,dataset_type,model_type, PE):
    
    root = "../dataset/benchmarks"
    mode = "test"

    cfg_path_dir = "../configs/model_cfg/QDAGer/{:s}/dir_{:s}.yaml".format(dataset_type,name)
    cfg_path_SH = "../configs/model_cfg/QDAGer/{:s}/SH_{:s}.yaml".format(dataset_type,name)

    dir_cfg = yaml_to_dotdict(cfg_path_dir)
    SH_cfg = yaml_to_dotdict(cfg_path_SH)

    dim_in = (dir_cfg.enc.in_dim if model_type == "dir" else SH_cfg.enc.in_dim)
    loader = DataLoader(
            root,
            name,
            dataset_type,
            mode,
            256,
            dim_in = dim_in,
            N_max=20,
            connect_correlator=True,
            need_degree=True,
            features= PE,
        )
    if PE not in ["corr_dyn","hk_pe","rw_pe"]:
        raise ValueError(" PE type not included")
    
    if dataset_type=="equal":
        model_path_dir = "../models/{:s}_eq_dir_{:s}.pt".format(name,PE)
        model_path_SH = "../models/{:s}_eq_SH_{:s}.pt".format(name,PE)

    elif dataset_type=="unequal":
        model_path_dir = "../models/{:s}_uneq_dir_{:s}.pt".format(name,PE)
        model_path_SH = "../models/{:s}_uneq_SH_{:s}.pt".format(name,PE)
    else : 
        raise ValueError("wrong dataset query")


    if model_type == "dir":
        state_dir = torch.load(model_path_dir, map_location="cpu")
        model = DirectQDAGerModel(dir_cfg)
        model.load_state_dict(state_dir)
        model.eval()
    elif model_type == "SH":
        state_SH = torch.load(model_path_SH, map_location="cpu")
        model = SinkhornQDAGerModel(SH_cfg)
        model.load_state_dict(state_SH)
        model.eval()
    else : 
        raise ValueError("wrong model query")

    preds = []
    LBs = []
    UBs = []

    for i in range(loader.num_batches):
        batch = loader.get_batch(i)
        preds.extend(model(batch).reshape(1,-1)[0].tolist())
        LBs.extend(batch.lb.reshape(1,-1)[0].tolist())
        UBs.extend(batch.ub.reshape(1,-1)[0].tolist())

    preds = torch.tensor(preds)
    lb = torch.tensor(LBs)
    ub = torch.tensor(UBs)
    losses = (torch.relu(lb - preds) ** 2 + torch.relu(preds - ub) ** 2)
    mean = losses.mean()
    n = losses.shape[0]
    SE = ((torch.sqrt((((losses-mean)**2).sum())/(n-1)))/math.sqrt(n)).item()

    return mean, SE

In [4]:
import warnings
warnings.filterwarnings("ignore")
for name in Names: 
    for dt in data_types:
        for mt in model_types:
            for pe in ["corr_dyn","hk_pe","rw_pe"]: 
                mean, SE = get_mean_and_SE(name,dt,mt,pe)
                print("{:s} | {:s} | {:s} | {:s} | {:f} ± {:f}".format(name,pe,dt,mt,mean,SE))
        print(10*"-")
    print(50*"-")

yeast | corr_dyn | equal | dir | 0.817095 ± 0.008453
yeast | hk_pe | equal | dir | 0.896038 ± 0.010530
yeast | rw_pe | equal | dir | 0.967061 ± 0.011195
yeast | corr_dyn | equal | SH | 0.699099 ± 0.007321
yeast | hk_pe | equal | SH | 0.716592 ± 0.007635
yeast | rw_pe | equal | SH | 1.534804 ± 0.109478
----------
yeast | corr_dyn | unequal | dir | 1.867281 ± 0.019223
yeast | hk_pe | unequal | dir | 2.053310 ± 0.022678
yeast | rw_pe | unequal | dir | 2.104445 ± 0.023076
yeast | corr_dyn | unequal | SH | 1.540571 ± 0.014839
yeast | hk_pe | unequal | SH | 1.721894 ± 0.019887
yeast | rw_pe | unequal | SH | 1.769014 ± 0.018555
----------
--------------------------------------------------
